# 06 - Stage 1 Sentence Embeddings

This notebook compares a pretrained sentence-transformer embedding model with the TF-IDF baseline. It preserves the grouped repeated-validation protocol and only uses existing 200/600/900 codes. Embedding inference requires the optional `sentence-transformers` package.

This is a Stretch/model-comparison notebook rather than the required baseline.

In [3]:
!pip install -U sentence-transformers

Defaulting to user installation because normal site-packages is not writeable

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: C:\Users\Kruti\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



  Using cached sentence_transformers-6.2.0-py3-none-any.whl.metadata (20 kB)
  Using cached transformers-5.22.1-py3-none-any.whl.metadata (31 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached huggingface_hub-2.4.0-py3-none-any.whl.metadata (16 kB)
  Using cached hf_xet-1.8.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached httpx2-2.16.0-py3-none-any.whl.metadata (10 kB)
  Using cached httpcore2-2.16.0-py3-none-any.whl.metadata (27 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
Using cached sentence_transformers-6.2.0-py3-none-any.whl (760 kB)
Using cached huggingface_hub-2.4.0-py3-none-any.whl (857 kB)
Using cached hf_xet-1.8.0-cp38-abi3-win_amd64.whl (3.9 MB)
Using cached httpx2-2.16.0-py3-none-any.whl (97 kB)
Using cached httpcore2-2.16.0-py3-none-any.whl (85 kB)
Using cached transformers-5.22.1-py3-none-any.whl (13.1 MB)
Using cached token

In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import StratifiedGroupKFold

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

WORKSPACE = Path.cwd()
PROCESSED_DIR = WORKSPACE / "data" / "processed"
MODEL_DIR = WORKSPACE / "models"
REPORT_DIR = WORKSPACE / "reports"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

messages_path = PROCESSED_DIR / "coded_stage_messages.csv"
if not messages_path.exists():
    raise FileNotFoundError("Run 01_coded_chat_data_loading.ipynb first.")

messages = pd.read_csv(messages_path, dtype={"category": "string"})
messages["text"] = messages["text"].fillna("")
messages["category"] = messages["category"].astype(str)

try:
    from sentence_transformers import SentenceTransformer
except ImportError as exc:
    raise ImportError("Install sentence-transformers to run this notebook.") from exc
MODEL_NAME = "all-MiniLM-L6-v2"
messages = messages.sort_values(["chat_id", "msg_idx"]).copy()
messages["previous_text"] = messages.groupby("chat_id")["text"].shift(1).fillna("")
messages["next_text"] = messages.groupby("chat_id")["text"].shift(-1).fillna("")
messages["model_text"] = (
    "previous: " + messages["previous_text"] + " target: " + messages["text"] + " next: " + messages["next_text"]
)

encoder = SentenceTransformer(MODEL_NAME)
embeddings = encoder.encode(
    messages["model_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Embedding shape:", embeddings.shape)

c:\Users\Kruti\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Kruti\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kruti\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer

Embedding shape: (10831, 384)


## Grouped embedding validation

Embedding rows are aligned to the original chat IDs before splitting. The same `chat_id` never appears in both sides of a fold.

In [2]:
y = messages["category"].to_numpy()
groups = messages["chat_id"].to_numpy()
n_splits = min(5, messages["chat_id"].nunique())
rows = []

for repeat in range(3):
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED + repeat)
    for fold, (train_index, test_index) in enumerate(splitter.split(embeddings, y, groups)):
        classifier = LogisticRegression(
            class_weight="balanced",
            solver="lbfgs",
            max_iter=2000,
            random_state=RANDOM_SEED,
        )
        classifier.fit(embeddings[train_index], y[train_index])
        predictions = classifier.predict(embeddings[test_index])
        precision, recall, f1, _ = precision_recall_fscore_support(
            y[test_index], predictions,
            labels=["200", "600", "900"],
            average="macro",
            zero_division=0,
        )
        rows.append({
            "repeat": repeat,
            "fold": fold,
            "macro_precision": precision,
            "macro_recall": recall,
            "macro_f1": f1,
        })

metrics = pd.DataFrame(rows)
summary = metrics[["macro_precision", "macro_recall", "macro_f1"]].agg(["mean", "std"])
display(metrics)
display(summary)

,repeat,fold,macro_precision,macro_recall,macro_f1
0,0,0,0.647791,0.664490,0.642431
1,0,1,0.701276,0.711841,0.700781
2,0,2,0.701232,0.726782,0.700326
3,0,3,0.699268,0.700962,0.699964
4,0,4,0.652079,0.681631,0.654129
5,1,0,0.647791,0.664490,0.642431
6,1,1,0.701276,0.711841,0.700781
7,1,2,0.701232,0.726782,0.700326
8,1,3,0.699268,0.700962,0.699964
9,1,4,0.652079,0.681631,0.654129


,macro_precision,macro_recall,macro_f1
mean,0.680329,0.697141,0.679526
std,0.025737,0.022753,0.026685


## Save embeddings, classifier, and metrics

In [3]:
final_classifier = LogisticRegression(
    class_weight="balanced",
    solver="lbfgs",
    max_iter=2000,
    random_state=RANDOM_SEED,
)
final_classifier.fit(embeddings, y)

joblib.dump(final_classifier, MODEL_DIR / "stage1_embedding_logistic.joblib")
np.save(PROCESSED_DIR / "stage1_embeddings.npy", embeddings)
metrics.to_csv(REPORT_DIR / "stage1_embedding_fold_metrics.csv", index=False)
summary.to_csv(REPORT_DIR / "stage1_embedding_metric_summary.csv")
messages[["chat_id", "msg_idx", "category"]].to_csv(REPORT_DIR / "stage1_embedding_alignment.csv", index=False)

print("Saved embedding Stage 1 artifacts.")

Saved embedding Stage 1 artifacts.
